# Notebook 48R2 - Binary Ensemble Test
Tests whether binary ensemble strategies outperform greyscale blending.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import cv2, numpy as np, pandas as pd, json
from pathlib import Path
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

In [ ]:
ROOT=Path("/content/drive/MyDrive/MusicAnnotationProject")
N45_ROOT=ROOT/"observed_print_classifier_v8_6_notebook45"
N46_ROOT=ROOT/"observed_print_classifier_v8_6_notebook46"
RUN46=max([p for p in N46_ROOT.iterdir() if p.is_dir()],key=lambda p:p.stat().st_mtime)
OUT=RUN46/"notebook48R2_binary_ensemble"
TABLES46=RUN46/"tables"
HYB=OUT/"candidates"
TAB=OUT/"tables"
FIGS=OUT/"figures"
for p in [OUT,HYB,TAB,FIGS]: p.mkdir(parents=True,exist_ok=True)
inv=pd.read_csv(RUN46/"tables"/"whole_page_candidate_inventory.csv")
print(RUN46)

In [ ]:
def ink(im):
 return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def edges(im):
 return cv2.Canny(im,70,170)>0

def f1_from_features(ref,pred,reg,tol=1):
 import numpy as np, cv2
 ref=ref&reg; pred=pred&reg
 if not ref.any(): return np.nan,np.nan,np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8)
 p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
 q=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
 return float(p),float(q),float(2*p*q/max(p+q,1e-12))

In [ ]:
pairs=inv[inv.restorer.isin(["R1_biharmonic","R3_learned_unet"])]
pairs=pairs.pivot_table(index=["page","source_group","arm"],columns="restorer",values="candidate",aggfunc="first").reset_index()
rows=[]
for _,r in pairs.iterrows():
 b=cv2.imread(str(r["R1_biharmonic"]),0)
 u=cv2.imread(str(r["R3_learned_unet"]),0)
 h50=((b.astype(float)+u.astype(float))/2).astype(np.uint8)
 _,otsu=cv2.threshold(h50,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
 and_img=((ink(b)&ink(u))*255).astype(np.uint8)
 or_img=((ink(b)|ink(u))*255).astype(np.uint8)
 products={"Hybrid50":h50,"Hybrid50_Otsu":otsu,"AND_Ensemble":and_img,"OR_Ensemble":or_img}
 for name,img in products.items():
  fp=HYB/f"{r.page}_{r.arm}_{name}.png"
  cv2.imwrite(str(fp),img)
  rows.append({"page":r.page,"source_group":r.source_group,"arm":r.arm,"restorer":name,"candidate":str(fp)})
ensemble_inventory=pd.DataFrame(rows)
ensemble_inventory.to_csv(TAB/"ensemble_inventory.csv",index=False)

In [ ]:
# Copy the page_clean/page_obs/page_auth reconstruction cells from 46R6 here.
# Then replace whole_candidates with ensemble_inventory and run the same whole-page scoring block.
print("Generated ensemble inventory:", len(ensemble_inventory))

In [ ]:
def latest(root,req):
 hits=[p for p in Path(root).rglob("*") if p.is_dir() and all((p/r).exists() for r in req)]
 return max(hits,key=lambda p:p.stat().st_mtime)
REQ45=["mask_lock.json","notebook45_verdict.json","source_record.json","tables/region_manifest.parquet","tables/mask_readiness_metrics.csv"]
N45=latest(N45_ROOT,REQ45)
source=json.loads((N45/"source_record.json").read_text())
regions=pd.read_parquet(N45/"tables/region_manifest.parquet")
print(N45)


In [ ]:
source_v82=Path(source["source_v82"])
clean_map={p.stem:p for p in (ROOT/"source"/"clean").iterdir()}
page_obs={p:cv2.imread(str(source_v82/"whole_pages"/p/"observed.png"),0) for p in regions.page.unique()}
page_clean={p:cv2.imread(str(clean_map[p]),0) for p in regions.page.unique()}


In [ ]:
source45=json.loads((N45/"source_record.json").read_text())
v75=Path(source45["v75"])
v75_status=pd.read_csv(v75/"tables/cache_status.csv")
v75_cache={str(r.page):Path(r.cache) for _,r in v75_status.iterrows()}
page_auth={}
for page in sorted(regions.page.unique()):
  with np.load(v75_cache[page]) as z:
    page_auth[page]=z["mask"]>0


In [ ]:
metric_rows=[]
for _,row in ensemble_inventory.iterrows():
 page=row.page
 clean=page_clean[page]
 obs=page_obs[page]
 auth=page_auth[page]
 pred=cv2.imread(str(row.candidate),0)
 diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32))
 _,sm=ssim(clean,pred,data_range=255,full=True)
 ip,ir,if1=f1_from_features(ink(clean),ink(pred),auth)
 _,_,ef1=f1_from_features(edges(clean),edges(pred),auth)
 metric_rows.append({"page":page,"source_group":row.source_group,"arm":row.arm,"restorer":row.restorer,"mae_authoritative":float(diff[auth].mean()),"ssim_authoritative":float(sm[auth].mean()),"ink_f1":if1,"edge_f1":ef1,"false_inserted_ink":int((ink(pred)&~ink(clean)&auth).sum()),"missing_ink":int((ink(clean)&~ink(pred)&auth).sum())})
ensemble_metrics=pd.DataFrame(metric_rows)
ensemble_metrics.to_csv(TAB/"ensemble_metrics.csv",index=False)


In [ ]:
parent=pd.read_csv(TABLES46/"whole_page_candidate_metrics.csv")
parent=parent[parent.restorer.isin(["R1_biharmonic","R3_learned_unet"])]
combined=pd.concat([parent,ensemble_inventory],ignore_index=True)
summary=combined.groupby("restorer",as_index=False).agg(false_inserted_ink=("false_inserted_ink","mean"),missing_ink=("missing_ink","mean"),mae_authoritative=("mae_authoritative","mean"),ink_f1=("ink_f1","mean"),edge_f1=("edge_f1","mean"))
summary.to_csv(TAB/"ensemble_summary.csv",index=False)
display(summary)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(summary.false_inserted_ink,summary.missing_ink,s=200)
for _,r in summary.iterrows(): plt.annotate(r.restorer,(r.false_inserted_ink,r.missing_ink))
plt.xlabel("False Inserted Ink")
plt.ylabel("Missing Ink")
plt.grid()
plt.savefig(FIGS/"ensemble_frontier.png",dpi=300,bbox_inches="tight")
plt.show()
